<a href="https://colab.research.google.com/github/NEO-Research-Group/quantuma-course/blob/main/QUBO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

We first install the Ocean SDK (D-Wave software).

In [ ]:
!pip install dwave-ocean-sdk networkx

Now we need to configure the environment (press 'enter' to use the default options). If you have an API key to use D-Wave systems, you can enter the key at this stage.

In [ ]:
!dwave setup

Import the required libraries.

In [11]:
from dimod import Binary, ExactSolver
from networkx import Graph, complete_graph, draw
from dwave.samplers import SimulatedAnnealingSampler, SteepestDescentSampler
from dwave.system import DWaveSampler, EmbeddingComposite, FixedEmbeddingComposite
from dwave.system.testing import MockDWaveSampler
from dwave.embedding.chain_strength import uniform_torque_compensation
from minorminer import find_embedding
import matplotlib.pyplot as plt

This function creates the QUBO expression from the graph.

In [12]:
def get_QUBO(G):
  nvars = len(G.nodes())
  var = [Binary(f'v{i}') for i in range (0,nvars)]
  qubo = -sum([(var[i]-var[j])**2 for i,j in G.edges()])
  return qubo

We define a function to get the sampler, which will try to solve the problem.

In [31]:
def get_sampler():
  #return ExactSolver()
  #return SimulatedAnnealingSampler()
  return SteepestDescentSampler()
  #return EmbeddingComposite(DWaveSampler())

Let's use this simple graph.

In [ ]:
G = Graph()
G.add_nodes_from([0,1,2,3,4])
G.add_edges_from([(0,1),(0,3),(1,2),(1,3),(2,4),(3,4)])
draw(G, with_labels=True)

We show the QUBO model for this graph:

In [ ]:
qubo = get_QUBO(G)
print(qubo)

We compute the solution using the sampler. This could be a Dwave quantum annealer if the API key was introduced during the setup stage of the SDK and the appropriate sampler was chosen.

In [ ]:
sampler = get_sampler()
result = sampler.sample(qubo, num_reads=100)
print(result)

## Embedding and chain strength

The QPU does not have a qubit for each variable connected to every other qubit: the qubits are arranged in a fixed graph (*Pegasus* in the Advantage systems). To solve a QUBO, each logical variable is mapped to a **chain** of physical qubits (this is the *minor embedding*). The qubits of a chain are joined by strong couplings so that they all take the same value. The strength of these couplings is the **`chain_strength`** parameter:

* If it is **too low**, the chains **break** (the qubits of a chain end up with different values) and the solution has to be repaired, usually by majority vote.
* If it is **too high**, the chain couplings dominate the problem. On the real QPU, the original QUBO coefficients are then compressed into a tiny range and the quality of the solutions gets worse.

To experiment without using QPU time, we use `MockDWaveSampler`. It has the same topology (Pegasus P16) and the same properties as an Advantage QPU, so the embedding and the chains are the same as on the real hardware. The quantum annealing itself is replaced by a classical sampler. We use a short simulated annealing run (few sweeps) to get noisy samples, a bit like the QPU. This is only an approximation: analog noise, control errors (ICE) and limited precision on the real QPU are not simulated.

In [ ]:
mock_qpu = MockDWaveSampler(
    topology_type='pegasus', topology_shape=[16],   # Advantage-like topology
    substitute_sampler=SimulatedAnnealingSampler(),
    substitute_kwargs={'num_sweeps': 20},           # few sweeps -> noisy, QPU-like samples
    exact_solver_cutoff=0,                          # never solve small problems exactly
    parameter_warnings=False)

# To use the real QPU, replace mock_qpu with DWaveSampler()
qpu = mock_qpu
print(f"Qubits: {len(qpu.nodelist)}, couplers: {len(qpu.edgelist)}")

The graph of the previous example is so small that it needs almost no chains. We use a complete graph with 16 nodes instead: every node is connected to every other node, so chains are needed. The max-cut of $K_n$ has $\lfloor n/2 \rfloor \cdot \lceil n/2 \rceil$ edges, so the optimum energy for $n=16$ is $-64$. You can change `n` or use any other graph.

In [ ]:
n = 16
G2 = complete_graph(n)
qubo2 = get_QUBO(G2)
print(f"Optimal energy: {-(n // 2) * ((n + 1) // 2)}")

We compute the embedding only once and reuse it with `FixedEmbeddingComposite`, so that the only difference between runs is the chain strength. If `chain_strength` is not given, Ocean computes a default value with `uniform_torque_compensation`.

In [ ]:
embedding = find_embedding(list(qubo2.quadratic), qpu.edgelist, random_seed=1)
chain_lengths = [len(chain) for chain in embedding.values()]
print(f"Logical variables: {len(embedding)}, physical qubits: {sum(chain_lengths)}")
print(f"Chain lengths: {sorted(chain_lengths)}")
print(f"Default chain strength: {uniform_torque_compensation(qubo2):.2f}")

embedded_sampler = FixedEmbeddingComposite(qpu, embedding)

Now **change `chain_strength`** and run the cell again. Try very low values (0.1, 0.5), medium values (1, 2, 4) and very high values (20, 100). Look at the fraction of broken chains (`chain_break_fraction` column) and at how often the optimum is found.

In [ ]:
chain_strength = 0.5   # <-- change me

result = embedded_sampler.sample(qubo2, num_reads=100, chain_strength=chain_strength)

energies = result.record.energy
counts = result.record.num_occurrences
print(f"Best energy: {result.first.energy}")
print(f"Mean energy: {(energies * counts).sum() / counts.sum():.2f}")
print(f"Mean chain break fraction: {(result.record.chain_break_fraction * counts).sum() / counts.sum():.3f}")
print(result.truncate(10))

Finally, we try a range of chain strengths and plot the fraction of broken chains and the mean energy for each value. Which value gives the best results? Is it close to the default value?

In [ ]:
strengths = [0.25, 0.5, 1, 1.5, 2, 3, 4, 8, 16, 32, 64]
break_fractions, mean_energies = [], []
for cs in strengths:
    r = embedded_sampler.sample(qubo2, num_reads=200, chain_strength=cs)
    w = r.record.num_occurrences
    break_fractions.append((r.record.chain_break_fraction * w).sum() / w.sum())
    mean_energies.append((r.record.energy * w).sum() / w.sum())

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(strengths, mean_energies, 'o-', color='tab:blue')
ax1.set_xscale('log')
ax1.set_xlabel('chain_strength')
ax1.set_ylabel('Mean energy (lower is better)', color='tab:blue')
ax2 = ax1.twinx()
ax2.plot(strengths, break_fractions, 's--', color='tab:red')
ax2.set_ylabel('Mean chain break fraction', color='tab:red')
ax1.axvline(uniform_torque_compensation(qubo2), color='gray', linestyle=':', label='default')
ax1.legend()
plt.show()

If you did not configure Dwave software above, you can always configure it using the following command.

In [ ]:
!dwave config create